# 02 - Error Correction Model (ECM) from ARDL

This notebook covers the **Error Correction Model (ECM)** derived from the ARDL
framework, including speed of adjustment, long-run coefficients, and diagnostic
tests using `chronobox`.

## Topics covered

1. From ARDL to ECM: the reparameterization
2. Speed of adjustment ($\alpha$) and long-run coefficients
3. Conditional vs unconditional ECM representation
4. Short-run dynamics vs long-run equilibrium
5. Diagnostics: Breusch-Godfrey, CUSUM, normality
6. ARDL vs Johansen for ECM estimation
7. Exercises

---

### From ARDL to ECM

Any ARDL(p, q₁, …, qₖ) model can be reparameterized as an ECM. Starting from:

$$y_t = c + \sum_{i=1}^{p} \phi_i \, y_{t-i} + \sum_{j=1}^{k} \sum_{l=0}^{q_j} \beta_{jl} \, x_{j,t-l} + u_t$$

We obtain the **error correction form**:

$$\Delta y_t = c + \underbrace{\pi_{yy} \, y_{t-1} + \boldsymbol{\pi}_{yx}' \, \mathbf{x}_{t-1}}_{\text{long-run (error correction)}}
  + \underbrace{\sum_{i=1}^{p-1} \gamma_i \, \Delta y_{t-i}
  + \sum_{j=1}^{k} \sum_{l=0}^{q_j-1} \delta_{jl} \, \Delta x_{j,t-l}}_{\text{short-run dynamics}} + u_t$$

where:
- $\pi_{yy} = -(1 - \sum \phi_i)$ is the **speed of adjustment** (should be negative for stability)
- $\boldsymbol{\pi}_{yx}$ captures the long-run levels effect of $\mathbf{x}$
- Long-run coefficients: $\theta_j = -\pi_{yx,j} / \pi_{yy}$
- $\gamma_i, \delta_{jl}$ are short-run adjustment coefficients

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
import sys
import os

from chronobox.models.ardl import ARDL
from chronobox.models.ecm import ECM

sys.path.insert(0, os.path.join("..", "utils"))
from plot_helpers import plot_ecm_adjustment, plot_long_run_relationship

%matplotlib inline
plt.rcParams["figure.dpi"] = 100
np.set_printoptions(precision=4, suppress=True)

print("All imports loaded successfully.")

## 1. Loading the Data

We use the synthetic ARDL dataset with mixed integration orders.
The true data-generating process has:
- Long-run equilibrium: $y = 1.5 + 0.6 \cdot x_1$
- Speed of adjustment: $\alpha = -0.25$
- x2 is I(0) with short-run effects only

In [ ]:
data_path = os.path.join("..", "data", "ardl_synthetic.csv")
df = pd.read_csv(data_path, parse_dates=["date"])

print(f"Shape: {df.shape}")
print(f"Period: {df['date'].iloc[0].date()} to {df['date'].iloc[-1].date()}")
print(df[["y", "x1", "x2", "x3"]].describe().round(4))

## 2. Step 1: Fit the ARDL Model

Before deriving the ECM, we first estimate the ARDL and confirm that a long-run
relationship exists via the bounds test.

In [ ]:
y = df["y"].values
x = df[["x1", "x2", "x3"]].values

# Fit ARDL with automatic lag selection
ardl = ARDL(max_p=4, max_q=4, criterion="aic")
ardl_result = ardl.fit(y, x)

print(f"Selected: ARDL({ardl_result.y_lags}, {', '.join(map(str, ardl_result.x_lags))})")
print(f"R² = {ardl_result.r_squared:.4f}")

# Bounds test to confirm cointegration
bt = ardl_result.bounds_test()
print(f"\nBounds test F = {bt['f_statistic']:.4f}")
print(f"Conclusion at 5%: {bt['conclusion']}")
print("\n→ Cointegration confirmed. We can proceed to ECM estimation.")

## 3. ECM Estimation

There are two ways to obtain the ECM in `chronobox`:

1. **From ARDL**: `ardl_result.to_ecm()` — converts the ARDL to ECM form
2. **Direct estimation**: `ECM(lags=p).fit(y, x)` — estimates ECM directly via OLS

Both give the same error correction representation:

$$\Delta y_t = c + \pi_{yy} \, y_{t-1} + \boldsymbol{\pi}_{yx}' \, \mathbf{x}_{t-1}
+ \text{short-run terms} + u_t$$

In [ ]:
# Method 1: Convert ARDL result to ECM
ecm_from_ardl = ardl_result.to_ecm()

# Method 2: Direct ECM estimation
ecm_direct = ECM(lags=2)
ecm_result = ecm_direct.fit(y, x)

print(ecm_result.summary())

## 4. Speed of Adjustment ($\pi_{yy}$)

The **speed of adjustment** parameter $\pi_{yy}$ (also called $\alpha$ or $\gamma$)
tells us how fast $y$ corrects back to equilibrium after a shock:

- $\pi_{yy} < 0$: **Error-correcting** — $y$ moves back toward equilibrium ✓
- $\pi_{yy} > 0$: **Explosive** — $y$ moves away from equilibrium ✗
- $|\pi_{yy}|$ close to 0: slow adjustment
- $|\pi_{yy}|$ close to 1: fast adjustment (half-life ≈ 1 period)

The **half-life** of a deviation from equilibrium is approximately:
$$\text{half-life} \approx \frac{\ln(0.5)}{\ln(1 + \pi_{yy})}$$

In [ ]:
alpha = ecm_result.speed_of_adjustment

print("Speed of Adjustment Analysis")
print("=" * 50)
print(f"pi_yy (speed of adjustment): {alpha:.4f}")
print(f"True DGP value:              -0.2500")

if alpha < 0:
    half_life = np.log(0.5) / np.log(1 + alpha)
    print(f"\nStatus: Error-correcting (negative) ✓")
    print(f"Half-life of disequilibrium: {half_life:.1f} quarters")
    print(f"  → After a shock, ~50% of the deviation is corrected in {half_life:.1f} periods")
    pct_per_q = abs(alpha) * 100
    print(f"  → Each quarter, ~{pct_per_q:.1f}% of the disequilibrium is corrected")
else:
    print("\nWarning: Positive speed of adjustment — system is explosive!")

In [ ]:
# Visualize the error correction dynamics
lr_const = 1.5  # DGP true value
lr_coef = ecm_result.long_run_coefficients[0]  # estimated for x1

fig = plot_ecm_adjustment(
    y=df["y"].values,
    x=df["x1"].values,
    long_run_const=lr_const,
    long_run_coef=lr_coef,
    dates=pd.DatetimeIndex(df["date"]),
    var_names=("y", "x1"),
    title="ECM Adjustment Dynamics",
)
plt.savefig(os.path.join("..", "outputs", "ecm_adjustment.png"), bbox_inches="tight")
plt.show()

print("Panel 1: y tracks the long-run equilibrium (dashed red) over time")
print("Panel 2: x1 drives the equilibrium path")
print("Panel 3: equilibrium error — fluctuates around zero (mean-reverting)")

## 5. Long-Run Coefficients

The long-run coefficients represent the equilibrium relationship:

$$y^* = \hat{c} + \theta_1 \, x_1 + \theta_2 \, x_2 + \theta_3 \, x_3$$

where $\theta_j = -\pi_{yx,j} / \pi_{yy}$.

In [ ]:
lr_coefs = ecm_result.long_run_coefficients
var_names = ["x1", "x2", "x3"]

print("Long-Run Coefficients")
print("=" * 50)
print(f"{'Variable':>10}  {'Estimated':>12}  {'True DGP':>12}")
print("-" * 50)

true_values = [0.60, 0.0, 0.0]  # DGP: only x1 enters the cointegrating relation
for name, est, true in zip(var_names, lr_coefs, true_values):
    print(f"{name:>10}  {est:>12.4f}  {true:>12.4f}")

print("=" * 50)
print(f"\npi_yy:  {ecm_result.pi_yy:.4f}")
print(f"pi_yx:  {ecm_result.pi_yx}")
print("\nThe long-run coef of x1 should be close to 0.6 (true DGP value).")
print("x2 (I(0)) and x3 (not cointegrated) have no true long-run equilibrium effect.")

## 6. Short-Run Dynamics

The **short-run coefficients** capture how changes (first differences) affect $\Delta y_t$:

- $\gamma_i$ (lagged $\Delta y$): persistence in changes
- $\delta_{jl}$ (lagged $\Delta x_j$): immediate dynamic effects of regressors

The short-run and long-run operate simultaneously:
- **Short-run**: $\Delta x$ affects $\Delta y$ within a few periods
- **Long-run**: deviations from equilibrium are corrected at speed $|\pi_{yy}|$

In [ ]:
print("Short-Run Dynamics")
print("=" * 50)
print(f"\nLagged Delta y coefficients (gamma_i):")
for i, g in enumerate(ecm_result.short_run_y, 1):
    print(f"  gamma_{i} = {g:.4f}")

print(f"\nLagged Delta x coefficients (delta_jl):")
sr_x = ecm_result.short_run_x
n_lags = ecm_result.short_run_x.shape[0] // 3 if len(sr_x) > 0 else 0
for i, d in enumerate(sr_x):
    j = i // max(n_lags, 1)
    l = i % max(n_lags, 1)
    print(f"  delta_{var_names[j]}_lag{l} = {d:.4f}")

print("\n--- Interpretation ---")
print("Short-run coefficients tell us the immediate dynamic response.")
print("Long-run coefficients tell us where the system settles in equilibrium.")
print(f"The system corrects ~{abs(ecm_result.speed_of_adjustment)*100:.1f}% of disequilibrium each period.")

## 7. Diagnostic Tests

To validate the ECM, we need to check:

1. **No serial correlation** (Breusch-Godfrey test)
2. **Parameter stability** (CUSUM test)
3. **Normal residuals** (Jarque-Bera / Shapiro-Wilk)

### 7.1 Breusch-Godfrey Test for Serial Correlation

Tests H₀: no serial correlation in residuals up to lag $m$.
Rejection suggests the model is misspecified (needs more lags).

In [ ]:
def breusch_godfrey_test(residuals, nobs, k_params, max_lag=4):
    """Breusch-Godfrey LM test for serial correlation."""
    n = len(residuals)
    results = []
    for m in range(1, max_lag + 1):
        # Auxiliary regression: e_t on lagged residuals + original regressors proxy
        y_aux = residuals[m:]
        X_aux = np.column_stack([
            np.ones(n - m),
            *[residuals[m - i:n - i] for i in range(1, m + 1)]
        ])
        beta_aux = np.linalg.lstsq(X_aux, y_aux, rcond=None)[0]
        fitted_aux = X_aux @ beta_aux
        r2_aux = 1 - np.sum((y_aux - fitted_aux)**2) / np.sum((y_aux - y_aux.mean())**2)
        lm_stat = (n - m) * r2_aux
        p_value = 1 - stats.chi2.cdf(lm_stat, df=m)
        results.append({"lag": m, "LM_stat": lm_stat, "p_value": p_value})
    return pd.DataFrame(results)


bg_results = breusch_godfrey_test(ecm_result.residuals, ecm_result.nobs, ecm_result.k_params)

print("Breusch-Godfrey Test for Serial Correlation")
print("H0: No serial correlation up to lag m")
print("=" * 50)
print(f"{'Lag':>5}  {'LM Stat':>10}  {'p-value':>10}  {'Decision':>14}")
print("-" * 50)
for _, row in bg_results.iterrows():
    dec = "Reject H0" if row["p_value"] < 0.05 else "Fail to reject"
    print(f"{int(row['lag']):>5}  {row['LM_stat']:>10.4f}  {row['p_value']:>10.4f}  {dec:>14}")

if all(bg_results["p_value"] > 0.05):
    print("\n✓ No evidence of serial correlation at 5% level.")
else:
    print("\n✗ Serial correlation detected — consider adding more lags.")

### 7.2 CUSUM Test for Parameter Stability

The **CUSUM** (Cumulative Sum of Recursive Residuals) test checks whether the
model parameters are stable over the sample period. If the CUSUM line crosses the
5% critical boundaries, the parameters are unstable (structural break possible).

In [ ]:
def cusum_test(residuals, k_params):
    """Compute CUSUM statistics and critical boundaries."""
    n = len(residuals)
    sigma = np.std(residuals, ddof=k_params)
    cusum = np.cumsum(residuals) / sigma
    
    # 5% critical boundaries: +/- a + b*t  (Brown, Durbin, Evans 1975)
    t = np.arange(1, n + 1)
    a = 0.948  # approximate for 5%
    boundary_upper = a * np.sqrt(n) + 2 * a * t / np.sqrt(n)
    boundary_lower = -boundary_upper
    
    return cusum, boundary_upper, boundary_lower


cusum, upper, lower = cusum_test(ecm_result.residuals, ecm_result.k_params)

fig, ax = plt.subplots(figsize=(12, 5))
t = np.arange(1, len(cusum) + 1)
ax.plot(t, cusum, color="steelblue", linewidth=1.5, label="CUSUM")
ax.plot(t, upper, "r--", linewidth=1.0, label="5% boundary")
ax.plot(t, lower, "r--", linewidth=1.0)
ax.fill_between(t, lower, upper, alpha=0.05, color="red")
ax.axhline(0, color="black", linewidth=0.5)
ax.set_xlabel("Observation")
ax.set_ylabel("CUSUM")
ax.set_title("CUSUM Test for Parameter Stability")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "ecm_cusum.png"), bbox_inches="tight")
plt.show()

within_bounds = np.all(np.abs(cusum) <= upper)
if within_bounds:
    print("✓ CUSUM within 5% boundaries — parameters appear stable.")
else:
    print("✗ CUSUM crosses boundaries — possible structural break.")

### 7.3 Normality of Residuals

OLS inference assumes normal errors. We test this with the **Jarque-Bera** test
(based on skewness and kurtosis) and the **Shapiro-Wilk** test.

In [ ]:
resid = ecm_result.residuals

# Jarque-Bera test
jb_stat, jb_pval = stats.jarque_bera(resid)

# Shapiro-Wilk test
sw_stat, sw_pval = stats.shapiro(resid)

print("Normality Tests on ECM Residuals")
print("=" * 50)
print(f"Skewness:  {stats.skew(resid):.4f}")
print(f"Kurtosis:  {stats.kurtosis(resid):.4f} (excess, normal = 0)")
print("-" * 50)
print(f"Jarque-Bera:  stat = {jb_stat:.4f}, p-value = {jb_pval:.4f}")
print(f"Shapiro-Wilk: stat = {sw_stat:.4f}, p-value = {sw_pval:.4f}")
print("-" * 50)

if jb_pval > 0.05 and sw_pval > 0.05:
    print("✓ Cannot reject normality at 5% level.")
else:
    print("✗ Normality rejected — consider robust standard errors.")

In [ ]:
# Residual plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Residual time series
axes[0].plot(resid, linewidth=0.8, color="steelblue")
axes[0].axhline(0, color="red", linestyle="--", linewidth=0.8)
axes[0].set_title("ECM Residuals")
axes[0].set_xlabel("Observation")
axes[0].set_ylabel("Residual")
axes[0].grid(True, alpha=0.3)

# Histogram
axes[1].hist(resid, bins=25, density=True, alpha=0.7, color="steelblue", edgecolor="black")
x_grid = np.linspace(resid.min(), resid.max(), 100)
axes[1].plot(x_grid, stats.norm.pdf(x_grid, resid.mean(), resid.std()),
             "r-", linewidth=2, label="Normal")
axes[1].set_title("Residual Distribution")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

# QQ plot
stats.probplot(resid, dist="norm", plot=axes[2])
axes[2].set_title("Q-Q Plot")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join("..", "outputs", "ecm_residual_diagnostics.png"), bbox_inches="tight")
plt.show()

## 8. PSS t-test on the ECM

In addition to the F-test (bounds test from Notebook 01), the PSS framework provides
a **t-test** on $\pi_{yy}$ in the ECM. This tests whether the error correction
coefficient is significantly different from zero.

In [ ]:
pss_t = ecm_result.bounds_test_pss()

print("PSS t-test on ECM")
print("=" * 50)
print(f"t-statistic (pi_yy): {pss_t['t_statistic']:.4f}")
print(f"pi_yy estimate:      {pss_t['pi_yy']:.4f}")
print(f"I(0) 5% bound:       {pss_t['lower_5']:.4f}")
print(f"I(1) 5% bound:       {pss_t['upper_5']:.4f}")
print(f"Conclusion:          {pss_t['conclusion']}")
print("\nNote: The t-test complements the F-test from Notebook 01.")
print("Both should agree on whether a long-run relationship exists.")

## 9. ARDL vs Johansen for ECM Estimation

Both approaches lead to an ECM, but through different paths:

| Aspect | ARDL → ECM | Johansen → VECM |
|--------|-----------|----------------|
| **Estimation** | Single equation (OLS) | System estimation (ML) |
| **Integration orders** | I(0), I(1), or mix | All must be I(1) |
| **Speed of adjustment** | One α (for y only) | α vector (one per variable) |
| **Cointegrating relations** | One (y as dependent) | Multiple possible (rank r) |
| **Small samples** | Better properties | Oversized tests, biased |
| **Exogeneity** | Assumes x weakly exogenous | No exogeneity assumption |

**Use ARDL → ECM when:**
- Variables have mixed I(0)/I(1) integration
- Small sample (T < 80–100)
- Clear dependent variable (single equation)
- Want to avoid pre-testing for unit roots

**Use Johansen → VECM when:**
- All variables confirmed I(1)
- Need to estimate multiple cointegrating relations
- System approach needed (no variable is clearly "dependent")
- Large sample available

## Exercicio

### Exercicio 1: ECM with US Macro Data

Using `us_macro_quarterly.csv`:
1. Set y = `gdp`, x = `[inflation, fed_funds]`
2. Fit an ARDL and convert to ECM
3. Interpret the speed of adjustment
4. Compute the half-life of disequilibrium
5. Run the three diagnostic tests
6. Is the error-correction mechanism significant?

In [ ]:
# --- Exercicio 1: Seu codigo aqui ---

us_df = pd.read_csv(os.path.join("..", "data", "us_macro_quarterly.csv"), parse_dates=["date"])

y_us = us_df["gdp"].values
x_us = us_df[["inflation", "fed_funds"]].values

# ARDL + ECM
ardl_us = ARDL(max_p=4, max_q=4, criterion="aic")
result_us = ardl_us.fit(y_us, x_us)
ecm_us = result_us.to_ecm()

print(ecm_us.summary())
print(f"\nSpeed of adjustment: {ecm_us.speed_of_adjustment:.4f}")
if ecm_us.speed_of_adjustment < 0:
    hl = np.log(0.5) / np.log(1 + ecm_us.speed_of_adjustment)
    print(f"Half-life: {hl:.1f} quarters")

# Diagnostics
bg = breusch_godfrey_test(ecm_us.residuals, ecm_us.nobs, ecm_us.k_params)
print(f"\nBreusch-Godfrey (lag 4): LM = {bg.iloc[-1]['LM_stat']:.4f}, p = {bg.iloc[-1]['p_value']:.4f}")

jb, jb_p = stats.jarque_bera(ecm_us.residuals)
print(f"Jarque-Bera: stat = {jb:.4f}, p = {jb_p:.4f}")

pss = ecm_us.bounds_test_pss()
print(f"\nPSS t-test: t = {pss['t_statistic']:.4f}, conclusion: {pss['conclusion']}")

### Exercicio 2: Effect of Lag Length on Speed of Adjustment

Using the synthetic dataset with y and x1 only:
1. Estimate ECM with lags = 1, 2, 3, 4
2. Compare the estimated speed of adjustment
3. Compute the long-run coefficient for each lag length
4. How sensitive are the results to the number of lags?

In [ ]:
# --- Exercicio 2: Seu codigo aqui ---

y_biv = df["y"].values
x_biv = df[["x1"]].values

print(f"{'Lags':>5}  {'pi_yy':>10}  {'LR coef(x1)':>14}  {'R²':>8}  {'AIC':>10}")
print("-" * 55)

for lag in [1, 2, 3, 4]:
    ecm_i = ECM(lags=lag).fit(y_biv, x_biv)
    print(f"{lag:>5}  {ecm_i.speed_of_adjustment:>10.4f}  {ecm_i.long_run_coefficients[0]:>14.4f}"
          f"  {ecm_i.r_squared:>8.4f}  {ecm_i.aic:>10.4f}")

print("\nTrue DGP: pi_yy = -0.25, LR coef(x1) = 0.60")
print("The estimates should be relatively stable across lag lengths.")

---

## Resumo

Neste notebook aprendemos:

1. O **ECM** e derivado do ARDL por reparametrizacao, separando dinamica de curto e longo prazo
2. O **speed of adjustment** ($\pi_{yy}$) mede a velocidade de correcao ao equilibrio
3. A **half-life** quantifica quanto tempo leva para corrigir metade de um desvio
4. Os **coeficientes de longo prazo** ($\theta_j$) medem a relacao de equilibrio
5. **Diagnosticos** sao essenciais: Breusch-Godfrey (correlacao serial), CUSUM (estabilidade), normalidade
6. O **ECM via ARDL** e preferivel ao Johansen VECM quando ha integracao mista ou amostra pequena

**Referências:**
- Pesaran, M. H., Shin, Y., & Smith, R. J. (2001). Bounds testing approaches. *JAPE*, 16(3).
- Engle, R. F., & Granger, C. W. J. (1987). Co-integration and error correction. *Econometrica*, 55(2).
- Brown, R. L., Durbin, J., & Evans, J. M. (1975). Techniques for testing the constancy of regression relationships over time. *JRSS-B*, 37(2).